# Altitude waypoint tracking — closed-loop validation

Validates `z_track.py` against the full 4-state truth model from
`../control/theory.md` §7, before anything is sent to the rig.

The controller does **not** use a linearization. The lift law
`z̈ = g·((f_r/f_h)² − 1)` inverts exactly as `f_r = f_h·√(1 + a_des/g)`, so the
outer loop is a PID producing an acceleration, clamped in acceleration space by
the torque budget, then inverted. See the plan for why the linearized
`2g/f_h` gain is the wrong tool here (100% error at rest, ±14% gain drift
across the band).

What this notebook checks:

1. The torque budget is self-consistent — `f_ceiling > f_hover`, i.e. the
   calibration actually permits flight.
2. `z` tracks the waypoints through the real cascade, where thrust responds to
   the robot's *actual* spin `ω`, not the commanded field frequency.
3. `sin δ` stays under `s_lim` for the whole run **including ramp ends**, which
   is where the lightly-damped (ζ ≈ 0.02) phase mode rings.

In [ ]:
import math, sys, pathlib
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "model" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "ai"))

import z_track
from z_track import TorqueLimits, ZTracker, load_waypoints, z_ref

lim = TorqueLimits()
times, heights = load_waypoints(ROOT / "ai" / "waypoints_z.json")

print(f"f_hover    = {lim.f_hover:.1f} Hz     f_stepout = {lim.f_stepout:.1f} Hz (measured knob)")
print(f"f_ceiling  = {lim.f_ceiling():.1f} Hz     s_lim     = {lim.s_lim}")
print(f"a_max      = {lim.a_max():.3f} m/s^2 ({lim.a_max()/lim.g:.2f} g)")
print(f"a_dot_max  = {lim.a_dot_max(lim.f_hover):.2f} m/s^3 at f_hover")
print(f"sin(delta) = {lim.sin_delta(lim.f_hover):.3f} at f_hover")
print(f"\nwaypoints: {list(zip(times, heights))}")

In [ ]:
# 4-state truth model (notes §7). Thrust responds to the ROBOT's spin omega,
# not the commanded field frequency u -- that gap is the whole point of
# simulating the cascade rather than trusting f_robot == f_field.
#
#   delta_dot = 2*pi*u - omega
#   omega_dot = (tau_max(u)*sin(delta) - k_drag*f_r*|f_r|) / I
#   z_dot     = w
#   w_dot     = g*((f_r/f_h)^2 - 1)          [pad is a UNILATERAL constraint]

TWO_PI = 2.0 * math.pi

def rhs(t, s, u, lim):
    delta, omega, z, w = s
    f_r = omega / TWO_PI
    w_dot = lim.g * ((f_r / lim.f_hover) ** 2 - 1.0)
    if z <= 0.0 and w_dot <= 0.0:          # resting on the pad: N carries the rest
        z_dot, w, w_dot = 0.0, 0.0, 0.0
    else:
        z_dot = w
    return [
        TWO_PI * u - omega,
        (lim.tau_max(u) * math.sin(delta) - lim.k_drag * f_r * abs(f_r)) / lim.i_robot,
        z_dot,
        w_dot,
    ]

RATE_HZ = 30.0                     # matches ai/hover_controller_runner.py
DT = 1.0 / RATE_HZ
T_END = float(times[-1]) + 4.0
# MaxStep resolves the ~18 Hz, Q~22 phase ringing (notes §5.5); without it the
# solver steps over the oscillation and sin(delta) looks artificially calm.
MAX_STEP = 2e-4

# Engage at the hover trim, on the pad (N = 0) -- this is what FLIGHT state
# looks like after main_flight.cpp finishes its 30 s open-loop spin-up ramp.
delta0 = math.asin(min(1.0, lim.sin_delta(lim.f_hover)))
s = np.array([delta0, TWO_PI * lim.f_hover, 0.0, 0.0])

trk = ZTracker(times, heights, lim)
log = {k: [] for k in ("t", "z", "z_ref", "w", "f_cmd", "f_robot", "sin_delta")}

for i in range(int(T_END / DT)):
    t = i * DT
    f_cmd = trk.step(t, z=float(s[2]), dt=DT, z_dot=float(s[3]))
    sol = solve_ivp(rhs, (t, t + DT), s, args=(f_cmd, lim),
                    max_step=MAX_STEP, rtol=1e-8, atol=1e-10)
    s = sol.y[:, -1]
    if s[2] <= 0.0:                 # inelastic landing
        s[2], s[3] = 0.0, 0.0
    f_r = s[1] / TWO_PI
    for k, v in zip(log, (t, s[2], z_ref(t, times, heights), s[3], f_cmd, f_r,
                          lim.k_drag * f_r * abs(f_r) / lim.tau_max(f_cmd))):
        log[k].append(v)

log = {k: np.asarray(v) for k, v in log.items()}
airborne = log["z"] > 1e-6
print(f"peak |sin(delta)| = {np.abs(log['sin_delta']).max():.3f}  (s_lim = {lim.s_lim})")
print(f"final z           = {log['z'][-1]*1000:.1f} mm")
print(f"f_cmd range       = {log['f_cmd'].min():.1f} .. {log['f_cmd'].max():.1f} Hz")
if airborne.any():
    err = (log["z"] - log["z_ref"])[airborne]
    print(f"airborne |z-z_ref|: mean {np.abs(err).mean()*1000:.1f} mm, max {np.abs(err).max()*1000:.1f} mm")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

ax[0].plot(log["t"], log["z_ref"] * 1000, "k--", lw=1.2, label="$z_{ref}$")
ax[0].plot(log["t"], log["z"] * 1000, lw=1.6, label="$z$")
ax[0].plot(times, heights * 1000, "o", ms=5, color="0.3", label="waypoints")
ax[0].set_ylabel("height [mm]")
ax[0].set_title("Altitude waypoint tracking via field frequency")

# Autoscaled to the signal on purpose: an f_ceiling line at 167 Hz would flatten
# a +-0.5 Hz trace into nothing. The torque margin is panel 3's job.
ax[1].axhline(lim.f_hover, color="0.7", ls=":", lw=1, label=f"$f_h$ = {lim.f_hover:.0f} Hz")
ax[1].plot(log["t"], log["f_cmd"], lw=1.4, label="$f_{cmd}$ (field)")
ax[1].plot(log["t"], log["f_robot"], lw=1.0, alpha=0.8, label="$f_{robot}$ (actual)")
ax[1].set_ylabel("frequency [Hz]")
ax[1].text(0.99, 0.06,
           f"$f_{{ceiling}}$ = {lim.f_ceiling():.1f} Hz (off scale)\n"
           f"used: {log['f_cmd'].max() - lim.f_hover:+.2f} Hz of "
           f"{lim.f_ceiling() - lim.f_hover:.1f} Hz available",
           transform=ax[1].transAxes, ha="right", va="bottom", fontsize=8, color="0.35")

ax[2].axhline(1.0, color="crimson", lw=1.2, label="step-out")
ax[2].axhline(lim.s_lim, color="crimson", ls="--", lw=1, label=f"$s_{{lim}}$ = {lim.s_lim}")
ax[2].plot(log["t"], log["sin_delta"], lw=1.0, color="darkorange")
ax[2].set_ylabel(r"torque ratio  $\sin\delta$")
ax[2].set_xlabel("time [s]")
ax[2].set_ylim(0, 1.1)

for a in ax:
    a.grid(alpha=0.3)
    a.legend(loc="best", fontsize=8)
fig.tight_layout()

out = ROOT / "model" / "z_tracking_sim.png"
fig.savefig(out, dpi=150)
print(f"saved {out}")
print(f"headroom used: {log['f_cmd'].max() - lim.f_hover:+.2f} Hz of "
      f"{lim.f_ceiling() - lim.f_hover:.1f} Hz -- this trajectory is gentle; "
      f"peak sin(delta) {np.abs(log['sin_delta']).max():.3f} vs s_lim {lim.s_lim}")